In [ ]:
import polars as pl

pl.Config.set_tbl_cols(99)
pl.Config.set_tbl_width_chars(999)
pl.Config.set_tbl_rows(999)

In [ ]:
df_batch_measurements = pl.scan_parquet("../data/batch_measurements.parquet")
df_measurements = pl.scan_parquet("../data/measurements.parquet")


def clean(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(pl.col("name").str.to_titlecase())

When was the first and last measurement of each bear taken?


In [ ]:
(
    df_measurements.group_by("name")
    .agg(
        first_measurement=pl.col("timestamp").min(),
        last_measurement=pl.col("timestamp").max(),
    )
    .collect()
)

For each lifestage group of polar bears and for each year, which polar bear was the most active (most amount of steps per day)?


In [ ]:
(
    df_batch_measurements.pipe(clean)
    .with_columns(year=pl.col("timestamp").dt.year())
    .with_columns(
        max_daily_steps=pl.col("daily_steps").max()
        # .over() is a window function, which computes an aggregation within groups (partitions)
        # but WITHOUT collapsing rows like group_by does.
        # It broadcasts the aggregated value back to every row in the partition
        .over("year", "life_stage")
    )
    .filter(pl.col("daily_steps") == pl.col("max_daily_steps"))
    .select("life_stage", "year", "name", "daily_steps")
    .sort("year", "life_stage")
    .collect()
)

For every year, figure out which polar bear was the heaviest.


In [ ]:
(
    df_batch_measurements.pipe(clean)
    .with_columns(year=pl.col("timestamp").dt.year())
    .with_columns(max_weight=pl.col("weight").max().over("year"))
    .filter(pl.col("weight") == pl.col("max_weight"))
    .select("year", "name", "weight")
    .collect()
)

Find out which bears were more anxious (higher blood pressure) than average the day after New Year's Eve (probably because of fireworks)?


In [ ]:
(
    df_measurements
    # Let's use the average of the blood pressure per polar bear as the baseline
    .with_columns(avg_blood_pressure=pl.col("blood_pressure").mean().over("name"))
    .filter((pl.col("timestamp").dt.month() == 1) & (pl.col("timestamp").dt.day() == 2))
    .filter(pl.col("blood_pressure") > pl.col("avg_blood_pressure"))
    .select("name", year=pl.col("timestamp").dt.year())
    .unique()
    .sort("year", "name")
    .collect()
)

Which polar bear has the highest risk of becoming a diabetic?  
(polar bears have a higher risk of becoming diabetic after going through a high blood sugar level episode.  
An episode is defined as a three-day or longer period of an average daily bgl of 200)


In [ ]:
(
    df_measurements
    # Look at groups of each bear, per day
    .group_by("name", date=pl.col("timestamp").dt.date())
    .agg(pl.col("blood_glucose").mean())
    #
    .sort("name", "date")
    # For each bear, identify runs of consecutive high blood glucose days by
    # 1) marking high blood glucose days
    .with_columns(high_bgl=pl.col("blood_glucose") > 200)
    # 2) assigning a run-length-encoding id to each consecutive block of high/normal days
    # This gives each block a unique id, essentially grouping them
    .with_columns(rle_id=pl.col("high_bgl").rle_id().over("name"))
    # 3) counting the number of days within each group of consecutive high/normal bgl days
    .with_columns(consecutive_days=pl.col("rle_id").cum_count().over("name", "rle_id"))
    # 4) marking the end of an episode: a high blood glucose day where the count of consecutive days is at least 3
    .with_columns(
        episode_end=pl.col("high_bgl")
        & (pl.col("consecutive_days") >= 3)
        & (
            pl.col("consecutive_days")
            == pl.max("consecutive_days").over("name", "rle_id")
        )
    )
    .with_columns(
        episode_duration=pl.when(pl.col("episode_end"))
        .then(pl.col("consecutive_days"))
        .otherwise(0)
    )
    # For each bear, sum up the number of episodes and their total length
    .group_by("name")
    .agg(
        nr_of_episodes=pl.col("episode_end").sum(),
        length_of_episodes=pl.col("episode_duration").sum(),
    )
    .sort("nr_of_episodes", "length_of_episodes", descending=True)
    .collect()
)